In [1]:
import os, numpy as np
from manzanas.unidades import cargar_unidad
from manzanas.pais import rutas_unidad
from manzanas.perfiles import cargar_modelo

cut, unidad = 13108, "personas"  # "viviendas", "personas", "hogares"
datos, inv = cargar_unidad(cut, unidad)
modelo = cargar_modelo(os.path.join(rutas_unidad(unidad)["modelos"], f"modelo_{cut}.npz"))

W = modelo["W"]               # (perfiles, manzanas)
ids = datos.manzanas_ids      # MANZENT de cada columna de W
mu = modelo["mu_p"]           # tamaño de cada perfil (filas)
# total de unidades por manzana (columnas); los modelos antiguos no guardan nu_m -> se toma de datos.nu
nu = modelo["nu_m"] if modelo["nu_m"] is not None else datos.nu
assert W.shape == (len(mu), len(nu)) and len(nu) == len(ids)
assert mu.sum() == nu.sum()
print(f"comuna {cut}, {unidad}: W {W.shape} ({W.dtype})")
print(f"nu desde {'modelo' if modelo['nu_m'] is not None else 'datos.nu'}; sum mu = {mu.sum():,}, sum nu = {nu.sum():,}")

comuna 13108, personas: W (40946, 469) (float32)
nu desde datos.nu; sum mu = 116,943, sum nu = 116,943


In [2]:
# Márgenes de W* frente a los enteros mu_p y nu_m (el ajuste paró en tol, así que no son exactos)
err_filas = np.abs(W.sum(axis=1) - mu)
err_cols = np.abs(W.sum(axis=0) - nu)
print(f"tol = {modelo['tol']}, tol_rel = {modelo['tol_rel']}, convergió = {modelo['convergio']}")
print(f"max |sum_m W - mu_p| = {err_filas.max():.4f}")
print(f"max |sum_p W - nu_m| = {err_cols.max():.4f}")
print(f"celdas fraccionales: {(W != np.floor(W)).sum():,} de {W.size:,}")

tol = 0.1, tol_rel = 0.0001, convergió = True
max |sum_m W - mu_p| = 0.0283
max |sum_p W - nu_m| = 0.0247
celdas fraccionales: 13,447,333 de 19,203,674


## Redondeo controlado de $W^*$

Buscamos $y \in \mathbb{Z}^{P \times M}$ con
$$\sum_m y_{pm} = \mu_p \;\; \forall p, \qquad \sum_p y_{pm} = \nu_m \;\; \forall m, \qquad \lfloor W^*_{pm} \rfloor \le y_{pm} \le \lceil W^*_{pm} \rceil.$$

La matriz de restricciones es la de un problema de transporte (totalmente unimodular), así que basta un **flujo máximo**: con $y = \lfloor W^* \rfloor + z$, $z \in \{0,1\}$ en las celdas fraccionales,

- fuente $\to p$ con capacidad $r_p = \mu_p - \sum_m \lfloor W^*_{pm} \rfloor$,
- $p \to m$ con capacidad 1 si $W^*_{pm} \notin \mathbb{Z}$,
- $m \to$ sumidero con capacidad $c_m = \nu_m - \sum_p \lfloor W^*_{pm} \rfloor$.

Si el flujo satura todos los arcos de la fuente, $y$ es un redondeo válido.

In [3]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_flow

def redondear(W, mu, nu, eps=1e-9):
    W = np.where(np.abs(W - np.round(W)) < eps, np.round(W), W)   # limpia ruido numérico
    F = np.floor(W).astype(np.int64)
    frac = (W - F) > eps
    mu = np.round(mu).astype(np.int64); nu = np.round(nu).astype(np.int64)
    r = mu - F.sum(1); c = nu - F.sum(0)
    assert (r >= 0).all() and (c >= 0).all() and r.sum() == c.sum()

    P, M = W.shape
    s, t = 0, P + M + 1
    pi, mi = np.nonzero(frac)
    filas = np.r_[np.full(P, s), 1 + pi, 1 + P + np.arange(M)]
    cols  = np.r_[1 + np.arange(P), 1 + P + mi, np.full(M, t)]
    cap   = np.r_[r, np.ones(len(pi), np.int64), c].astype(np.int32)
    G = csr_matrix((cap, (filas, cols)), shape=(t + 1, t + 1))

    res = maximum_flow(G, s, t)
    assert res.flow_value == r.sum(), "no saturó: revisa márgenes de W*"
    z = np.asarray(res.flow[1 + pi, 1 + P + mi]).ravel()
    Y = F.copy(); Y[pi, mi] += z
    assert (Y.sum(1) == mu).all() and (Y.sum(0) == nu).all()
    return Y

Y = redondear(W, mu, nu)
print(f"Y {Y.shape}, total {Y.sum():,}")
print("filas = mu_p:", (Y.sum(axis=1) == mu).all(), "| columnas = nu_m:", (Y.sum(axis=0) == nu).all())
print("floor <= Y <= ceil:", ((Y >= np.floor(W)) & (Y <= np.ceil(W))).all())
print(f"max |Y - W*| = {np.abs(Y - W).max():.4f}, celdas redondeadas hacia arriba: {(Y > np.floor(W)).sum():,}")


Y (40946, 469), total 116,943
filas = mu_p: True | columnas = nu_m: True
floor <= Y <= ceil: True


max |Y - W*| = 1.0000, celdas redondeadas hacia arriba: 90,145


## Diagnóstico: el redondeo por flujo rompe los marginales por bloque

El flujo cumple $\mu_p$, $\nu_m$ y las cotas, pero no ve los marginales $\nu_{kcm}$ (bloque $k$, categoría $c$, manzana $m$). Como $W^*$ está muy repartido, en cada manzana hay que poner miles de "+1" y cualquier subconjunto sirve: los cambios se acumulan dentro de una categoría.

In [4]:
from manzanas.perfiles import errores_modelo, codigos_bloque, indicadora

eW = errores_modelo(inv, modelo)
eY = errores_modelo(inv, dict(modelo, W=Y.astype(float)))
print(f"{'bloque':16s} {'W*':>8s} {'Y flujo':>8s}")
for b in modelo["bloques"]:
    print(f"{b:16s} {eW[b]:8.3f} {eY[b]:8.0f}")

# la manzana con mayor error en edad
j = modelo["bloques"].index("edad")
nu_k = codigos_bloque(inv, "edad")[1]
A_k = indicadora(modelo["perfiles"][:, j], nu_k.shape[0])
pW, pY = A_k @ W.astype(float), A_k @ Y.astype(float)
m = np.nanargmax(np.nanmax(np.abs(pY - nu_k), axis=0))
print(f"\nmanzana {ids[m]} (nu = {nu[m]}), edad: censo / W* / Y")
for c in range(nu_k.shape[0]):
    print(f"  cat {c}: {nu_k[c, m]:6.0f} {pW[c, m]:8.1f} {pY[c, m]:6.0f}")

bloque                 W*  Y flujo
edad                0.016     2338
sexo                0.001     1563
inmigrante          0.033     2219
nacionalidad        0.003     2170
pueblo              0.001      352
afro                0.001      359
lengua              0.000      108
religion            0.025      430
dif_ver             0.002      267
dif_oir             0.001      194
dif_mover           0.002      417
dif_cogni           0.001      326
dif_cuidado         0.001      272
dif_comunic         0.001      218
discapacidad        0.003      658
estado_civil        0.034      521
asist_parv          0.008       31
asist_basica        0.018      143
asist_media         0.002      174
asist_superior      0.003      162
cine                0.015      702
analfabetismo       0.000       98
fuerza_trabajo      0.553     1005
cise                0.043     1007
ciuo                0.018      966
caenes              0.010      937
transporte          0.000      999

manzana 13108061002

## Muestreo de la pizarra: un punto extremo con $\mathbb{E}[Y] = W^*$

El flujo máximo y el muestreo resuelven **el mismo PL** (filas $\mu_p$, columnas $\nu_m$, cotas $\lfloor W^*\rfloor \le y \le \lceil W^*\rceil$); la diferencia es qué punto extremo se elige. El flujo entrega uno fijo y sesgado (el orden de los perfiles). El procedimiento de la pizarra escribe $W^* = \sum_i \lambda_i v_i$ como combinación convexa de puntos extremos y sortea $v_i$ con probabilidad $\lambda_i$:

1. $\lfloor W^*\rfloor \le Y \le \lceil W^*\rceil$ $\quad$ 2. $\mathbb{E}[Y] = W^*$ $\quad$ 3. $Y\mathbf{1} = \mu$, $Y^\top\mathbf{1} = \nu$

Los marginales por bloque no son restricciones, pero son lineales en $Y$: $\mathbb{E}\big[\sum_{p:\,x_{pk}=c} Y_{pm}\big] = \sum_{p:\,x_{pk}=c} W^*_{pm} \approx \nu_{kcm}$. Se cumplen **en esperanza**.

Sin calcular la descomposición, el *dependent rounding* (Gandhi, Khuller, Parthasarathy y Srinivasan, 2006) sortea directamente un punto extremo con esa distribución: busca un ciclo de celdas fraccionales (par, el grafo es bipartito), suma $+\alpha/-\alpha$ alternado con la dirección al azar de modo que la esperanza no cambie, y repite hasta que todo es entero. Además las celdas de una misma manzana quedan negativamente correlacionadas, así que cada marginal de bloque se concentra alrededor de su esperanza (cotas tipo Chernoff).

Requisito: $W^*$ debe cumplir $\mu_p$ y $\nu_m$ exactamente; antes se hace IPF solo en filas y columnas (cambio máximo ~0,001).

In [5]:
from manzanas import redondeo as rd

Wa, it = rd.ajustar_margenes(W, mu, nu)
print(f"ajuste de márgenes: {it} iteraciones, max |Wa - W*| = {np.abs(Wa - W).max():.4f}, "
      f"max |filas - mu| = {np.abs(Wa.sum(1) - mu).max():.1e}, max |columnas - nu| = {np.abs(Wa.sum(0) - nu).max():.1e}")

Ym, info = rd.muestrear(W, mu, nu, seed=0, W_ajustada=Wa)
print(f"muestra (semilla 0): {info['aristas']:,} celdas fraccionales, {info['ciclos']:,} ciclos, {info['segundos']:.0f} s")
print("filas = mu_p:", (Ym.sum(1) == mu).all(), "| columnas = nu_m:", (Ym.sum(0) == nu).all(),
      "| floor <= Y <= ceil:", ((Ym >= np.floor(Wa)) & (Ym <= np.ceil(Wa))).all())

ajuste de márgenes: 14 iteraciones, max |Wa - W*| = 0.0012, max |filas - mu| = 6.1e-12, max |columnas - nu| = 3.2e-12


muestra (semilla 0): 13,447,334 celdas fraccionales, 13,404,788 ciclos, 34 s
filas = mu_p: True | columnas = nu_m: True | floor <= Y <= ceil: True


In [6]:
# Comparación por bloque: error sobre las celdas (categoría, manzana) observadas
eb_W = rd.errores_bloque(inv, modelo, W)
eb_F = rd.errores_bloque(inv, modelo, Y)     # flujo máximo
eb_M = rd.errores_bloque(inv, modelo, Ym)    # muestreo
print(f"{'':15s} {'max |error|':>23s}   {'error medio':>21s}   {'% celdas |err|<=1':>19s}")
print(f"{'bloque':15s} {'W*':>6s} {'flujo':>7s} {'muestreo':>8s}   {'W*':>5s} {'flujo':>6s} {'muestreo':>8s}   {'flujo':>6s} {'muestreo':>9s}")
for b in modelo["bloques"]:
    print(f"{b:15s} {eb_W[b]['max']:6.2f} {eb_F[b]['max']:7.0f} {eb_M[b]['max']:8.0f}   "
          f"{eb_W[b]['medio']:5.2f} {eb_F[b]['medio']:6.2f} {eb_M[b]['medio']:8.2f}   "
          f"{eb_F[b]['pct_<=1']:6.1f} {eb_M[b]['pct_<=1']:9.1f}")
for nombre, eb in [("flujo", eb_F), ("muestreo", eb_M)]:
    print(f"{nombre:9s}: peor bloque {max(v['max'] for v in eb.values()):.0f}, "
          f"error medio promedio {np.mean([v['medio'] for v in eb.values()]):.2f}")
print(f"sum |Y - W*|: flujo {np.abs(Y - W).sum():,.0f} | muestreo {np.abs(Ym - W).sum():,.0f}")

                            max |error|             error medio     % celdas |err|<=1
bloque              W*   flujo muestreo      W*  flujo muestreo    flujo  muestreo
edad              0.02    2338        2    0.00  28.23     0.04     23.6      99.8
sexo              0.00    1563        6    0.00  42.39     0.64     36.0      86.4
inmigrante        0.03    2219       17    0.00  76.17     1.56     11.3      57.5
nacionalidad      0.00    2170       10    0.00  75.79     1.48     11.8      59.5
pueblo            0.00     352       12    0.00  15.42     1.79     21.2      56.9
afro              0.00     359       10    0.00   6.09     0.61     69.7      84.3
lengua            0.00     108       11    0.00   5.26     0.93     60.3      77.2
religion          0.02     430       13    0.00  35.86     1.99      5.8      46.7
dif_ver           0.00     267       12    0.00   9.44     1.55     24.3      61.0
dif_oir           0.00     194       11    0.00   5.05     1.03     37.3      75.3
d

In [7]:
# Varias semillas: variabilidad entre muestras y chequeo de E[Y] = W* (el promedio de n muestras debe acercarse
# a W*, y con él los marginales: el error del promedio baja ~ 1/sqrt(n))
semillas = range(1, 6)
muestras = [Ym] + [rd.muestrear(W, mu, nu, seed=s, W_ajustada=Wa)[0] for s in semillas]
ebs = [rd.errores_bloque(inv, modelo, y) for y in muestras]
for i, eb in enumerate(ebs):
    print(f"semilla {i}: peor bloque {max(v['max'] for v in eb.values()):4.0f} | "
          f"error medio promedio {np.mean([v['medio'] for v in eb.values()]):.3f}")
for n in [1, 2, len(muestras)]:
    Yprom = np.mean(muestras[:n], axis=0)
    eb = rd.errores_bloque(inv, modelo, Yprom)
    print(f"promedio de {n} muestras: error medio promedio {np.mean([v['medio'] for v in eb.values()]):.3f}, "
          f"max |prom - W*| = {np.abs(Yprom - Wa).max():.2f}")

semilla 0: peor bloque   34 | error medio promedio 1.251
semilla 1: peor bloque   42 | error medio promedio 1.257
semilla 2: peor bloque   52 | error medio promedio 1.254
semilla 3: peor bloque   72 | error medio promedio 1.243
semilla 4: peor bloque   40 | error medio promedio 1.242
semilla 5: peor bloque   44 | error medio promedio 1.251


promedio de 1 muestras: error medio promedio 1.251, max |prom - W*| = 1.00


promedio de 2 muestras: error medio promedio 0.896, max |prom - W*| = 1.00


promedio de 6 muestras: error medio promedio 0.522, max |prom - W*| = 0.86


## Redondeo con restricciones por categoría (MIP)

Notación: $x_{pk}$ categoría del perfil $p$ en el bloque $k$; $\nu_{kcm}$ marginal del censo; $\mathcal{O}_k$ celdas $(c,m)$ observadas (sin NaN).

Con $y_{pm} = \lfloor W^*_{pm}\rfloor + z_{pm}$:
$$
\begin{aligned}
\min\ & \textstyle\sum_k \sum_{(c,m)\in\mathcal{O}_k} (d^+_{kcm} + d^-_{kcm}) \\
\text{s.a. } & \textstyle\sum_m y_{pm} = \mu_p && \forall p \\
& \textstyle\sum_p y_{pm} = \nu_m && \forall m \\
& \textstyle\sum_{p:\,x_{pk}=c} y_{pm} - d^+_{kcm} + d^-_{kcm} = \nu_{kcm} && \forall k,\ (c,m)\in\mathcal{O}_k \\
& z_{pm}\in\{0,1\} \text{ si } (p,m) \text{ es libre}, \quad z_{pm} = 0 \text{ si no}, \quad d^\pm \ge 0
\end{aligned}
$$

$\mu_p$ y $\nu_m$ son duras; los bloques son blandos (sin solución dura garantizada: redondeo controlado multidimensional). Con todas las celdas fraccionales serían 13 millones de binarias, así que solo son **libres** las celdas con parte fraccional $\ge \tau$ más las $L$ mayores de cada fila (para que $\mu_p, \nu_m$ sigan siendo factibles). El MIP se resuelve con HiGHS en segundo plano (`scripts/correr_redondeo.py`); aquí se arma, se revisa y se carga el resultado.

In [8]:
from manzanas import redondeo as rd

F, frac, r, c = rd.partes(W, mu, nu)
print(f"a llenar con z: {r.sum():,} unidades | celdas fraccionales: {(frac > rd.EPS).sum():,}")
print(f"{'tau':>6s} {'L':>2s} {'libres':>11s}  mu/nu factible")
for tau, L in [(0.1, 0), (0.05, 0), (0.02, 0), (0.01, 0), (0.05, 3), (0.02, 3)]:
    pi, mi = np.nonzero(rd.celdas_libres(frac, tau, L))
    _, ok = rd.flujo(r, c, pi, mi)
    print(f"{tau:6g} {L:2d} {len(pi):11,}  {ok}")

a llenar con z: 90,145 unidades | celdas fraccionales: 13,213,648
   tau  L      libres  mu/nu factible
   0.1  0     122,916  False
  0.05  0     286,474  False


  0.02  0     800,313  False
  0.01  0   1,539,450  True


  0.05  3     336,395  True


  0.02  3     801,262  True


In [9]:
TAU, L = 0.05, 3
mip = rd.construir(W, mu, nu, inv, modelo, tau=TAU, L=L)
print(f"tau = {TAU}, L = {L}: A {mip['A'].shape}, nnz {mip['A'].nnz:,}")
print(f"binarias z: {mip['n_z']:,} | restricciones de bloque (blandas): {mip['B']:,}")

z0, obj0 = rd.solucion_inicial(mip)
Y0 = rd.matriz_Y(mip, z0)
e0 = errores_modelo(inv, dict(modelo, W=Y0.astype(float)))
print(f"solución inicial (flujo con perfiles en orden aleatorio): sum|d| = {obj0:,.0f}")
print("filas = mu_p:", (Y0.sum(1) == mu).all(), "| columnas = nu_m:", (Y0.sum(0) == nu).all())
print("máximo error de bloque:", max(e0[b] for b in modelo["bloques"]))

tau = 0.05, L = 3: A (91424, 436413), nnz 9,734,014
binarias z: 336,395 | restricciones de bloque (blandas): 50,009


solución inicial (flujo con perfiles en orden aleatorio): sum|d| = 360,948
filas = mu_p: True | columnas = nu_m: True
máximo error de bloque: 739.0
